# Notebook 01 — Data Schema & Loading

## Dataset: KKBox Music Streaming Churn Prediction

### File inventory (actual files on disk)
| File | Role | Notes |
|------|------|-------|
| `train_v2.csv` | Churn labels (March 2017) | **Primary label file** |
| `train.csv` | Churn labels (February 2017) | Secondary — earlier cohort |
| `members_v3.csv` | Customer demographics | v3 supersedes v1/v2 |
| `transactions.csv` | Subscription history up to end-2016 | 1.7 GB — load via DuckDB |
| `transactions_v2.csv` | Subscription history Jan–Mar 2017 | 115 MB — load via DuckDB |
| `user_logs.csv` | Daily listening logs pre-Feb 2017 | **30.5 GB** — DuckDB only |
| `user_logs_v2.csv` | Daily listening logs Mar 2017 | 1.4 GB — DuckDB only |

**Key deviation from original Kaggle README**: column is `msno` not `customer_id`.  
Versioned files are date-range splits — we UNION ALL both versions for complete history.

In [1]:
import pandas as pd
import numpy as np
import duckdb
import os

DATA_DIR = "../data"

def file_info(fname):
    path = os.path.join(DATA_DIR, fname)
    size_mb = os.path.getsize(path) / 1e6
    return path, size_mb

print("Data directory:", os.path.abspath(DATA_DIR))
print("Files on disk:")
for f in sorted(os.listdir(DATA_DIR)):
    path, mb = file_info(f)
    print(f"  {f:35s}  {mb:>9.1f} MB")

Data directory: C:\Users\h11la\OneDrive\Documents\00 Portfolio\Customer Analytics ML Pipeline\retail-clv-churn-prediction\data
Files on disk:
  members_v3.csv                           427.9 MB
  train.csv                                 46.7 MB
  train_v2.csv                              45.6 MB
  transactions.csv                        1729.3 MB
  transactions_v2.csv                      115.4 MB
  user_logs.csv                          30514.1 MB
  user_logs_v2.csv                        1431.5 MB


## Section 1 — Small files via Pandas (train_v2, train, members_v3)

In [2]:
def describe_pandas(fname, label=""):
    path, size_mb = file_info(fname)
    df = pd.read_csv(path)
    print(f"\n{'='*60}")
    print(f"  {fname}  ({size_mb:.1f} MB)  {label}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]:,} rows  x  {df.shape[1]} cols")
    print(f"\nColumns, dtypes, null counts:")
    null_counts = df.isnull().sum()
    null_pct    = (null_counts / len(df) * 100).round(1)
    summary = pd.DataFrame({"dtype": df.dtypes, "nulls": null_counts, "null_%": null_pct})
    print(summary.to_string())
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())
    return df

train_v2  = describe_pandas("train_v2.csv",  label="[PRIMARY LABELS — March 2017]")
train     = describe_pandas("train.csv",      label="[SECONDARY LABELS — Feb 2017]")
members   = describe_pandas("members_v3.csv", label="[MEMBER DEMOGRAPHICS]")


  train_v2.csv  (45.6 MB)  [PRIMARY LABELS — March 2017]
Shape: 970,960 rows  x  2 cols

Columns, dtypes, null counts:
          dtype  nulls  null_%
msno        str      0     0.0
is_churn  int64      0     0.0

First 3 rows:
                                           msno  is_churn
0  ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=         1
1  f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=         1
2  zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=         1



  train.csv  (46.7 MB)  [SECONDARY LABELS — Feb 2017]
Shape: 992,931 rows  x  2 cols

Columns, dtypes, null counts:
          dtype  nulls  null_%
msno        str      0     0.0
is_churn  int64      0     0.0

First 3 rows:
                                           msno  is_churn
0  waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=         1
1  QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=         1
2  fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=         1



  members_v3.csv  (427.9 MB)  [MEMBER DEMOGRAPHICS]
Shape: 6,769,473 rows  x  6 cols

Columns, dtypes, null counts:
                        dtype    nulls  null_%
msno                      str        0     0.0
city                    int64        0     0.0
bd                      int64        0     0.0
gender                    str  4429505    65.4
registered_via          int64        0     0.0
registration_init_time  int64        0     0.0

First 3 rows:
                                           msno  city  bd gender  registered_via  registration_init_time
0  Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=     1   0    NaN              11                20110911
1  +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=     1   0    NaN               7                20110914
2  cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=     1   0    NaN              11                20110915


## Section 2 — Large files via DuckDB (transactions + user_logs, both versions UNION ALL)

In [3]:
con = duckdb.connect()

# Register all large CSVs as DuckDB views
con.execute(f"""
    CREATE OR REPLACE VIEW transactions AS
    SELECT *, 'v1' AS source FROM read_csv_auto('{DATA_DIR}/transactions.csv')
    UNION ALL
    SELECT *, 'v2' AS source FROM read_csv_auto('{DATA_DIR}/transactions_v2.csv')
""")

con.execute(f"""
    CREATE OR REPLACE VIEW user_logs AS
    SELECT *, 'v1' AS source FROM read_csv_auto('{DATA_DIR}/user_logs.csv')
    UNION ALL
    SELECT *, 'v2' AS source FROM read_csv_auto('{DATA_DIR}/user_logs_v2.csv')
""")

def describe_duckdb(view_name, files):
    print(f"\n{'='*60}")
    print(f"  {view_name}  [DuckDB UNION ALL: {', '.join(files)}]")
    print(f"{'='*60}")

    # Row count per source
    counts = con.execute(f"SELECT source, COUNT(*) AS n FROM {view_name} GROUP BY source ORDER BY source").df()
    print("Row counts by source file:")
    print(counts.to_string(index=False))
    total = counts["n"].sum()
    print(f"Total rows: {total:,}")

    # Schema
    schema = con.execute(f"DESCRIBE {view_name}").df()
    print(f"\nSchema:")
    print(schema[["column_name", "column_type"]].to_string(index=False))

    # First 3 rows
    sample = con.execute(f"SELECT * FROM {view_name} LIMIT 3").df()
    print(f"\nFirst 3 rows:")
    print(sample.to_string())

    # Null counts (sample 100k rows to keep it fast)
    null_sql = ", ".join(
        [f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}"
         for c in sample.columns if c != "source"]
    )
    nulls = con.execute(f"SELECT {null_sql} FROM {view_name} USING SAMPLE 100000").df()
    print(f"\nNull counts (sampled 100k rows):")
    print(nulls.T.rename(columns={0: "nulls"}).to_string())

describe_duckdb("transactions", ["transactions.csv", "transactions_v2.csv"])
describe_duckdb("user_logs",    ["user_logs.csv",    "user_logs_v2.csv"])


  transactions  [DuckDB UNION ALL: transactions.csv, transactions_v2.csv]


Row counts by source file:
source        n
    v1 21547746
    v2  1431009
Total rows: 22,978,755

Schema:
           column_name column_type
                  msno     VARCHAR
     payment_method_id      BIGINT
     payment_plan_days      BIGINT
       plan_list_price      BIGINT
    actual_amount_paid      BIGINT
         is_auto_renew      BIGINT
      transaction_date      BIGINT
membership_expire_date      BIGINT
             is_cancel      BIGINT
                source     VARCHAR



First 3 rows:
                                           msno  payment_method_id  payment_plan_days  plan_list_price  actual_amount_paid  is_auto_renew  transaction_date  membership_expire_date  is_cancel source
0  YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=                 41                 30              129                 129              1          20150930                20151101          0     v1
1  AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=                 41                 30              149                 149              1          20150930                20151031          0     v1
2  UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=                 41                 30              129                 129              1          20150930                20160427          0     v1



Null counts (sampled 100k rows):
                        nulls
msno                      0.0
payment_method_id         0.0
payment_plan_days         0.0
plan_list_price           0.0
actual_amount_paid        0.0
is_auto_renew             0.0
transaction_date          0.0
membership_expire_date    0.0
is_cancel                 0.0

  user_logs  [DuckDB UNION ALL: user_logs.csv, user_logs_v2.csv]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Row counts by source file:
source         n
    v1 392106543
    v2  18396362
Total rows: 410,502,905

Schema:
column_name column_type
       msno     VARCHAR
       date      BIGINT
     num_25      BIGINT
     num_50      BIGINT
     num_75      BIGINT
    num_985      BIGINT
    num_100      BIGINT
    num_unq      BIGINT
 total_secs      DOUBLE
     source     VARCHAR



First 3 rows:
                                           msno      date  num_25  num_50  num_75  num_985  num_100  num_unq  total_secs source
0  rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=  20150513       0       0       0        0        1        1     280.335     v1
1  rxIP2f2aN0rYNp+toI0Obt/N/FYQX8hcO1fTmmy2h34=  20150709       9       1       0        0        7       11    1658.948     v1
2  yxiEWwE9VR5utpUecLxVdQ5B7NysUPfrNtGINaM2zA8=  20150105       3       3       0        0       68       36   17364.956     v1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Null counts (sampled 100k rows):
            nulls
msno          0.0
date          0.0
num_25        0.0
num_50        0.0
num_75        0.0
num_985       0.0
num_100       0.0
num_unq       0.0
total_secs    0.0


## Section 3 — Data Quality Checks

In [4]:
print("=== QC 1: members_v3 — invalid birthdates (bd) ===")
total_members = len(members)
invalid_bd = members[(members["bd"] <= 0) | (members["bd"] > 120)].shape[0]
print(f"  Total members     : {total_members:,}")
print(f"  Invalid bd (<=0 or >120): {invalid_bd:,}  ({invalid_bd/total_members*100:.1f}%)")

print("\n=== QC 2: transactions — actual_amount_paid > plan_list_price ===")
overpaid = con.execute("""
    SELECT COUNT(*) AS n
    FROM transactions
    WHERE actual_amount_paid > plan_list_price
      AND plan_list_price > 0
""").fetchone()[0]
total_tx = con.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
print(f"  Total transactions: {total_tx:,}")
print(f"  Overpaid rows     : {overpaid:,}  ({overpaid/total_tx*100:.3f}%)")
if overpaid > 0:
    print("  ⚠️  FLAG: overpaid rows exist — investigate before using plan_list_price")

print("\n=== QC 3: user_logs — rows where total_secs = 0 ===")
zero_secs = con.execute("SELECT COUNT(*) FROM user_logs WHERE total_secs = 0").fetchone()[0]
total_logs = con.execute("SELECT COUNT(*) FROM user_logs").fetchone()[0]
print(f"  Total log rows    : {total_logs:,}")
print(f"  Zero total_secs   : {zero_secs:,}  ({zero_secs/total_logs*100:.2f}%)")

print("\n=== QC 4: msno overlap — train_v2 vs members_v3 ===")
train_msno   = set(train_v2["msno"])
members_msno = set(members["msno"])
in_both      = train_msno & members_msno
only_train   = train_msno - members_msno
only_members = members_msno - train_msno
print(f"  Unique msno in train_v2   : {len(train_msno):,}")
print(f"  Unique msno in members_v3 : {len(members_msno):,}")
print(f"  In both                   : {len(in_both):,}")
print(f"  Only in train_v2 (no member info): {len(only_train):,}")
print(f"  Only in members_v3 (not labelled): {len(only_members):,}")

print("\n=== QC 5: churn rate in train_v2 vs train ===")
print(f"  train_v2 churn rate: {train_v2['is_churn'].mean()*100:.2f}%  (n={len(train_v2):,})")
print(f"  train    churn rate: {train['is_churn'].mean()*100:.2f}%  (n={len(train):,})")

=== QC 1: members_v3 — invalid birthdates (bd) ===


  Total members     : 6,769,473
  Invalid bd (<=0 or >120): 4,540,858  (67.1%)

=== QC 2: transactions — actual_amount_paid > plan_list_price ===


  Total transactions: 22,978,755
  Overpaid rows     : 202  (0.001%)
  ⚠️  FLAG: overpaid rows exist — investigate before using plan_list_price

=== QC 3: user_logs — rows where total_secs = 0 ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Total log rows    : 410,502,905
  Zero total_secs   : 0  (0.00%)

=== QC 4: msno overlap — train_v2 vs members_v3 ===


  Unique msno in train_v2   : 970,960
  Unique msno in members_v3 : 6,769,473
  In both                   : 860,967
  Only in train_v2 (no member info): 109,993
  Only in members_v3 (not labelled): 5,908,506

=== QC 5: churn rate in train_v2 vs train ===
  train_v2 churn rate: 8.99%  (n=970,960)
  train    churn rate: 6.39%  (n=992,931)
